In [ ]:
train = pd.read_csv(
    'data/train.csv',
    parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'],
)
test = pd.read_csv(
    'data/test.csv',
    parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'],
)
events = pd.read_csv('data/events.csv.gz', parse_dates=['event_ts'])

Для признаков использую только события, которые были доступны внутри окна конкретной cookie. События после `window_end_ts` исключаются, чтобы не допустить утечки будущей информации.

In [ ]:
def events_in_window(events, meta):
    """Оставляем только события, доступные внутри окна наблюдения cookie. события после window_end_ts
    не должны влиять ни на признаки, ни на предсказание.
    """
    ev = events.merge(
        meta[['cookie_id', 'window_start_ts', 'window_end_ts']],
        on='cookie_id',
        how='inner',
        validate='many_to_one',
    )
    return ev[
        (ev.event_ts >= ev.window_start_ts)
        & (ev.event_ts < ev.window_end_ts)
    ].copy()


ev_tr = events_in_window(events, train)
ev_te = events_in_window(events, test)


Baseline оставлен из quickstart: Random Forest на двух простых признаках — числе событий и числе уникальных объявлений. Нужен как понятная точка для сравнений.

In [ ]:
def basic_features(ev, meta):
    """Baseline из quickstart: объём событий и число уникальных item_id."""
    g = ev.groupby('cookie_id')
    f = pd.DataFrame({
        'n_events': g.size(),
        'item_nunique': g.item_id.nunique(),
    })
    f = meta[['cookie_id']].merge(f.reset_index(), on='cookie_id', how='left')
    return f.fillna(0)


Xtr_basic = basic_features(ev_tr, train)
Xte_basic = basic_features(ev_te, test)
ytr = train.target.values

is_valid = train.window_start_ts.ge('2026-04-17').values
basic_cols = ['n_events', 'item_nunique']

baseline = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=3,
    random_state=0,
    n_jobs=-1,
)
baseline.fit(Xtr_basic.loc[~is_valid, basic_cols], ytr[~is_valid])
p_va_baseline = baseline.predict_proba(Xtr_basic.loc[is_valid, basic_cols])[:, 1]

print(
    'baseline P@R0.7 на поздней валидации:',
    round(precision_at_recall(ytr[is_valid], p_va_baseline), 4),
)

Признаки строятся отдельно для каждой cookie и не используют `target`.

Использую:
- объём и типы активности;
- интервалы между событиями и активные периоды;
- разнообразие категорий, локаций и запросов;
- поисковое поведение;
- обобщённые характеристики платформы и user-agent;
- характеристики координат курсора.

Полные дубликаты событий удаляются. `NaN` не заполняются средним: HistGradientBoosting умеет работать с ними напрямую, а наличие пропуска может быть информативным.

In [ ]:
EVENT_NAMES = [
    'search_results_view',
    'item_view',
    'photo_swipe',
    'seller_page_view',
    'contact_phone_show',
    'contact_chat_open',
    'contact_message_sent',
    'favorite_add',
    'login',
    'captcha_shown',
]


def ua_family(ua):
    """Обобщаем user_agent, не используя полную строку как категорию."""
    s = str(ua).lower()
    for pattern, family in [
        ('python-requests', 'requests'),
        ('scrapy', 'scrapy'),
        ('node-fetch', 'node_fetch'),
        ('curl', 'curl'),
        ('headless', 'headless'),
        ('avito/', 'avito_app'),
        ('yabrowser', 'yandex'),
        ('edg/', 'edge'),
        ('firefox', 'firefox'),
        ('chrome', 'chrome'),
        ('safari', 'safari'),
    ]:
        if pattern in s:
            return family
    return 'other'


def entropy(s):
    counts = s.value_counts(dropna=True).to_numpy(dtype=float)
    if len(counts) == 0:
        return np.nan
    p = counts / counts.sum()
    return float(-(p * np.log(p)).sum())


def build_features(ev, meta):
    """Строим поведенческие агрегаты отдельно для каждой cookie.

    Мы не используем target при построении признаков и не считаем статистики
    по другим cookie. Поэтому признаки train/test строятся одинаково.
    """
    # Точные полные дубликаты считаем повторной записью одного события.
    # Удаляем только полностью совпадающие строки, не схлопывая похожие события.
    ev = ev.drop_duplicates().copy()

    ev['platform_clean'] = (
        ev.platform
        .str.strip()
        .str.lower()
        .replace({'desktop': 'web', 'iphone': 'ios'})
    )
    ev['ua_family'] = ev.user_agent.apply(ua_family)
    ev['ua_script'] = ev.ua_family.isin(
        ['requests', 'scrapy', 'node_fetch', 'curl', 'headless']
    )
    ev['ua_mobile'] = ev.user_agent.str.contains(
        'Android|iPhone|iPad|Mobile',
        case=False,
        regex=True,
    )
    ev['hour'] = ev.event_ts.dt.hour
    ev['minute'] = ev.event_ts.dt.floor('min')
    ev = ev.sort_values(['cookie_id', 'event_ts'], kind='stable')

    records = []

    for cookie, e in ev.groupby('cookie_id', sort=False):
        n = len(e)
        t = e.event_ts.astype('int64').to_numpy() / 1e9
        gaps = np.diff(t)
        positive_gaps = gaps[gaps > 0]
        span = t[-1] - t[0]

        f = {
            'cookie_id': cookie,
            'count_events': n,
            'time_span_s': span,
            'time_active_hours': e.hour.nunique(),
            'time_active_minutes': e.minute.nunique(),
            'time_hour_entropy': entropy(e.hour),
            'time_first_hour': e.hour.iloc[0],
            'time_last_hour': e.hour.iloc[-1],
            'time_night_share': e.hour.lt(6).mean(),
            'time_peak_minute_share': e.minute.value_counts().iloc[0] / n,
            'time_peak_hour_share': e.hour.value_counts().iloc[0] / n,
            'time_rate_per_minute': (n - 1) * 60 / span if span > 0 else np.nan,
            'time_sessions_30min': 1 + np.sum(gaps > 1800),
            'time_sessions_5min': 1 + np.sum(gaps > 300),
            'ua_script_share': e.ua_script.mean(),
            'ua_mobile_share': e.ua_mobile.mean(),
            'ua_nunique': e.user_agent.nunique(),
            'ua_family_nunique': e.ua_family.nunique(),
            'platform_nunique': e.platform_clean.nunique(),
        }

        # Платформа.
        for platform in ['web', 'android', 'ios']:
            f[f'platform_{platform}_share'] = e.platform_clean.eq(platform).mean()

        # Состав действий.
        for name in EVENT_NAMES:
            count = e.event_name.eq(name).sum()
            f[f'event_{name}_count'] = count
            f[f'event_{name}_share'] = count / n

        f['event_entropy'] = entropy(e.event_name)

        # Временные интервалы между событиями.
        if len(gaps):
            f['time_gap_mean'] = gaps.mean()
            f['time_gap_std'] = gaps.std()
            f['time_gap_cv'] = gaps.std() / gaps.mean() if gaps.mean() > 0 else np.nan

            for q, label in [
                (0, 'min'),
                (0.10, 'q10'),
                (0.25, 'q25'),
                (0.50, 'median'),
                (0.75, 'q75'),
                (0.90, 'q90'),
                (1, 'max'),
            ]:
                f[f'time_gap_{label}'] = np.quantile(gaps, q)

            for limit in [0, 1, 2, 5, 10, 30, 60, 300]:
                f[f'time_gap_le_{limit}_share'] = (gaps <= limit).mean()

            f['time_gap_mode_share'] = (
                pd.Series(gaps).value_counts().iloc[0] / len(gaps)
            )
            f['time_gap_entropy'] = entropy(pd.Series(gaps))
            f['time_gap_unique_share'] = len(np.unique(gaps)) / len(gaps)

            positive_order = gaps > 0
            names = e.event_name.to_numpy()
            f['event_repeat_share'] = (
                (names[1:][positive_order] == names[:-1][positive_order]).mean()
                if positive_order.any()
                else np.nan
            )

        if len(positive_gaps):
            f['time_positive_gap_cv'] = (
                positive_gaps.std() / positive_gaps.mean()
            )
            f['time_positive_gap_median'] = np.median(positive_gaps)

        # Разнообразие объектов и поискового поведения.
        for col in [
            'item_id',
            'item_category',
            'item_location',
            'seller_type',
            'search_query',
            'search_page',
        ]:
            s = e[col].dropna()
            counts = s.value_counts()
            f[f'{col}_nunique'] = len(counts)
            f[f'{col}_unique_share'] = len(counts) / len(s) if len(s) else np.nan
            f[f'{col}_entropy'] = entropy(s)
            f[f'{col}_top_share'] = counts.iloc[0] / len(s) if len(s) else np.nan
            f[f'{col}_missing_share'] = e[col].isna().mean()

        # Числовые характеристики поиска и координат.
        for col in ['search_page', 'pointer_x', 'pointer_y']:
            s = e[col].dropna()
            f[f'{col}_mean'] = s.mean()
            f[f'{col}_std'] = s.std()
            f[f'{col}_min'] = s.min()
            f[f'{col}_max'] = s.max()

        search = e[e.event_name.eq('search_results_view')]
        f['search_query_length_mean'] = search.search_query.str.len().mean()
        f['search_deep_share'] = (
            search.search_page.gt(5).mean() if len(search) else np.nan
        )

        if len(search) > 1:
            valid_search_step = (
                (search.event_ts.diff().dt.total_seconds() > 0)
                & search.search_query.eq(search.search_query.shift())
            )
            page_diff = search.search_page.diff()[valid_search_step]
            f['search_page_next_share'] = (
                page_diff.eq(1).mean() if len(page_diff) else np.nan
            )
            f['search_page_increase_share'] = (
                page_diff.gt(0).mean() if len(page_diff) else np.nan
            )

        pointer = e.dropna(subset=['pointer_x', 'pointer_y'])
        f['pointer_available_share'] = len(pointer) / n
        f['pointer_unique_share'] = (
            len(pointer[['pointer_x', 'pointer_y']].drop_duplicates()) / len(pointer)
            if len(pointer)
            else np.nan
        )
        f['pointer_zero_share'] = (
            ((pointer.pointer_x == 0) & (pointer.pointer_y == 0)).mean()
            if len(pointer)
            else np.nan
        )

        if len(pointer) > 1:
            valid_pointer_step = pointer.event_ts.diff().dt.total_seconds() > 0
            distance = np.sqrt(
                pointer.pointer_x.diff() ** 2 + pointer.pointer_y.diff() ** 2
            )[valid_pointer_step]
            f['pointer_distance_mean'] = distance.mean()
            f['pointer_stationary_share'] = (
                distance.eq(0).mean() if len(distance) else np.nan
            )

        # Отношения действий к просмотрам объявлений.
        for name in [
            'photo_swipe',
            'favorite_add',
            'contact_phone_show',
            'contact_message_sent',
        ]:
            f[f'ratio_{name}_to_item_views'] = (
                f[f'event_{name}_count']
                / max(f['event_item_view_count'], 1)
            )

        f['ratio_search_to_item_views'] = (
            f['event_search_results_view_count']
            / max(f['event_item_view_count'], 1)
        )

        records.append(f)

    features = pd.DataFrame(records)

    # Возвращаем все cookie, включая те, у которых нет событий в окне.
    features = meta[['cookie_id']].merge(features, on='cookie_id', how='left')
    features['age_days'] = (
        (meta.window_start_ts - meta.cookie_created_at).dt.total_seconds() / 86400
    ).to_numpy()
    features['no_events'] = features['count_events'].isna().astype(int)

    # HistGradientBoosting умеет работать с NaN напрямую. Поэтому не заполняем
    # пропуски средним: отсутствие значения само по себе может быть сигналом.
    numeric_cols = features.columns.difference(['cookie_id'])
    features[numeric_cols] = features[numeric_cols].replace(
        [np.inf, -np.inf], np.nan
    )

    return features


Xtr = build_features(ev_tr, train)
Xte = build_features(ev_te, test)

feature_cols = [c for c in Xtr.columns if c != 'cookie_id']
# Train и test должны иметь абсолютно одинаковый набор признаков.
assert feature_cols == [c for c in Xte.columns if c != 'cookie_id']
print('число признаков:', len(feature_cols))

Временная кросс-валидация

Случайный `train_test_split` / `StratifiedKFold` здесь не подходят: они смешали бы прошлое и будущее. Поэтому использую последовательные временные folds. Поздний период с 17 апреля оставляю для отдельной диагностической проверки.

In [ ]:
# Random/StratifiedKFold смешал бы прошлое и будущее и дал бы слишком
# оптимистичную оценку. Поэтому используем два последовательных временных fold.
# Дни с 17 апреля оставляем для отдельной поздней проверки.

is_late_valid = train.window_start_ts.ge('2026-04-17').values
is_dev = ~is_late_valid

train_dev = train.loc[is_dev].reset_index(drop=True)
X_dev = Xtr.loc[is_dev, feature_cols].reset_index(drop=True)
y_dev = ytr[is_dev]

folds = []
for valid_start, valid_end in [
    ('2026-04-11', '2026-04-14'),
    ('2026-04-14', '2026-04-17'),
]:
    train_idx = np.flatnonzero(
        train_dev.window_start_ts.lt(valid_start).to_numpy()
    )
    valid_idx = np.flatnonzero(
        (
            train_dev.window_start_ts.ge(valid_start)
            & train_dev.window_start_ts.lt(valid_end)
        ).to_numpy()
    )
    folds.append((train_idx, valid_idx))

Подбор параметров идёт по официальной метрике `Precision @ Recall >= 0.70`, а не по accuracy.

Полный Grid Search перебирает 162 комбинации. В финальном воспроизводимом запуске он выключен, потому что лучшие параметры уже найдены и зафиксированы в коде.

In [ ]:
param_grid = {
    'max_leaf_nodes': [3, 7, 15],
    'min_samples_leaf': [10, 20, 40],
    'l2_regularization': [0, 5, 20],
    'learning_rate': [0.03, 0.05],
    'max_iter': [300, 500, 800],
}

# GridSearchCV оптимизирует именно официальную метрику кейса, а не accuracy.
scorer = make_scorer(
    precision_at_recall,
    response_method='predict_proba',
)

if RUN_GRID_SEARCH:
    model = HistGradientBoostingClassifier(
        early_stopping=False,
        random_state=RANDOM_SEED,
    )

    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring=scorer,
        cv=folds,
        n_jobs=-1,
        refit=True,
        verbose=1,
        error_score='raise',
    )
    grid.fit(X_dev, y_dev)

    best_params = grid.best_params_
    print('лучшие параметры:', best_params)
    print('средний CV P@R0.7:', round(grid.best_score_, 4))

    pd.DataFrame(grid.cv_results_).to_csv('grid_results.csv', index=False)
    with open('best_params.json', 'w', encoding='utf-8') as f:
        json.dump(best_params, f, ensure_ascii=False, indent=2)
else:
    # Эти параметры заранее получены тем же GridSearchCV на двух временных folds.
    # Фиксируем их здесь, чтобы повторный запуск не требовал 162 конфигураций.
    best_params = BEST_PARAMS.copy()
    print('используем параметры, ранее выбранные GridSearchCV:', best_params)

Модель обучается на более ранних данных и проверяется на последних размеченных днях — по той же временной логике, что и в quickstart.

In [ ]:
best_model = HistGradientBoostingClassifier(
    **best_params,
    early_stopping=False,
    random_state=RANDOM_SEED,
)

best_model.fit(Xtr.loc[~is_late_valid, feature_cols], ytr[~is_late_valid])
p_va = best_model.predict_proba(
    Xtr.loc[is_late_valid, feature_cols]
)[:, 1]

print(
    'итоговый P@R0.7 на поздней валидации:',
    round(precision_at_recall(ytr[is_late_valid], p_va), 4),
)
print(
    'доля ботов (уровень константы):',
    round(ytr[is_late_valid].mean(), 4),
)

После выбора конфигурации модель обучается на всём train. В `submission.csv` сохраняется вероятность класса ботов (`predict_proba`), без фиксированного порога: порог перебирает проверяющая система.

Перед сохранением код проверяет длину файла, уникальность и порядок `cookie_id`, отсутствие нечисловых значений и диапазон score `[0, 1]`.

In [ ]:
# После выбора модели обучаем её на всём доступном train. Порог не применяем:
# проверяющая система сама перебирает пороги, поэтому сохраняем predict_proba.
best_model.fit(Xtr[feature_cols], ytr)

sub = pd.DataFrame({
    'cookie_id': Xte.cookie_id,
    'score': best_model.predict_proba(Xte[feature_cols])[:, 1],
})

assert len(sub) == len(test)
assert sub.cookie_id.is_unique
assert sub.cookie_id.tolist() == test.cookie_id.tolist()
assert np.isfinite(sub.score).all()
assert sub.score.between(0, 1).all()

sub.to_csv('submission.csv', index=False)
print('submission.csv сохранён')
print(sub.head())